<a href="https://colab.research.google.com/github/itsayeshaqamar/flyrank-mlinternship-ayesha/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/itsayeshaqamar/flyrank-mlinternship-ayesha/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# ML-08 data loading
import pandas as pd
import numpy as np

file_path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/data_0.parquet"
)

# Only load columns needed for ML-08
needed_columns = [
    "report_date",
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_users",
    "ga4_engaged_sessions",
    "ga4_total_engagement_sec",
    "sessions_organic",
    "sessions_direct",
    "sessions_referral",
    "sessions_social",
    "sessions_paid",
    "sessions_ai",
    "scroll_events"
]

df = pd.read_parquet(
    file_path,
    columns=needed_columns
)

df["report_date"] = pd.to_datetime(df["report_date"])

print("Loaded shape:", df.shape)
print("Date range:", df["report_date"].min(), "to", df["report_date"].max())

Loaded shape: (9841378, 17)
Date range: 2026-03-01 00:00:00 to 2026-03-31 00:00:00


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## 1. Method choice and why

I will use a **Random Forest Regressor** for this modeling lane.

The target is observed `sessions_organic`, which represents organic sessions for a content page on a daily observation.

Random Forest fits this lane because the relationship between search, traffic, engagement, and organic sessions may not be purely linear. It can capture nonlinear relationships while still allowing feature importance to be inspected.

I chose regression because the warehouse does not contain an observed `refresh_needed` or `refresh_priority` label. Using the Week-4 baseline score as the target would make the model learn the rule rather than provide an independent test.

The goal is therefore to estimate observed organic performance from available signals and use the result as **directional analysis and decision-support**, not as proof that a page needs a refresh.

The model will be compared with the Week-4 baseline using the same page-level evaluation data and a ranking-oriented metric.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-08 Section 1
# Confirm the modeling target and available feature fields

target = "sessions_organic"

features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_users",
    "ga4_engaged_sessions",
    "ga4_total_engagement_sec",
    "sessions_direct",
    "sessions_referral",
    "sessions_social",
    "sessions_paid",
    "sessions_ai",
    "scroll_events"
]

print("Target:", target)
print("Number of features:", len(features))

print("\nAll target/features exist:",
      all(col in df.columns for col in [target] + features))

Target: sessions_organic
Number of features: 14

All target/features exist: True


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## 2. Split design

I will use a **time-aware split**.

The earlier observations in March 2026 will be used for training, while the later observations will be held out for testing.

This is more honest than a random split because the task is based on observations over time. A random split could mix observations from the same page across train and test and make performance look stronger than it would be in a forward-looking setting.

The split is based on `report_date`, not on the target value.

No future observations are used to train the model.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-08 Section 2
# Time-aware split

unique_dates = sorted(df["report_date"].dropna().unique())

split_index = int(len(unique_dates) * 0.80)

train_dates = unique_dates[:split_index]
test_dates = unique_dates[split_index:]

train_end = train_dates[-1]
test_start = test_dates[0]

train_df = df[df["report_date"].isin(train_dates)]

test_df = df[df["report_date"].isin(test_dates)]

print("Training period:", train_dates[0], "to", train_end)
print("Testing period:", test_start, "to", test_dates[-1])

print("\nFull training rows:", len(train_df))
print("Full testing rows:", len(test_df))

Training period: 2026-03-01 00:00:00 to 2026-03-24 00:00:00
Testing period: 2026-03-25 00:00:00 to 2026-03-31 00:00:00

Full training rows: 7548489
Full testing rows: 2292889


In [3]:
# Create a manageable modeling sample
# Sampling is performed separately within the time-based split.

TRAIN_SAMPLE_SIZE = 200_000
TEST_SAMPLE_SIZE = 100_000

if len(train_df) > TRAIN_SAMPLE_SIZE:
    train_model = train_df.sample(
        n=TRAIN_SAMPLE_SIZE,
        random_state=42
    )
else:
    train_model = train_df.copy()

if len(test_df) > TEST_SAMPLE_SIZE:
    test_model = test_df.sample(
        n=TEST_SAMPLE_SIZE,
        random_state=42
    )
else:
    test_model = test_df.copy()

print("Model training rows:", len(train_model))
print("Model testing rows:", len(test_model))

Model training rows: 200000
Model testing rows: 100000


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

## 3. Train + compare vs my baseline

I will train the Random Forest on the earlier part of March and evaluate it on the later held-out observations.

Because the Week-4 baseline is a page-priority score while the model predicts organic sessions, I will aggregate the model's test predictions to the content-page level.

For each page, the model produces an average predicted organic-session value over the test period.

I will compare this ML ranking with the Week-4 baseline ranking using **Spearman rank correlation**.

This is a ranking comparison rather than a claim that either ranking represents ground-truth refresh priority.

The model is evaluated on observations that were not used for training.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-08 Section 3
# Train Random Forest on the manageable time-aware sample

from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error

target = "sessions_organic"

features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_users",
    "ga4_engaged_sessions",
    "ga4_total_engagement_sec",
    "sessions_direct",
    "sessions_referral",
    "sessions_social",
    "sessions_paid",
    "sessions_ai",
    "scroll_events"
]

# -----------------------------
# Prepare data
# -----------------------------

X_train = train_model[features]
X_test = test_model[features]

y_train = train_model[target].fillna(0)
y_test = test_model[target].fillna(0)

# -----------------------------
# Handle missing values
# -----------------------------

imputer = SimpleImputer(strategy="median")

X_train_imp = imputer.fit_transform(X_train)
X_test_imp = imputer.transform(X_test)

print("Training matrix:", X_train_imp.shape)
print("Testing matrix:", X_test_imp.shape)

# -----------------------------
# Train a smaller Random Forest
# -----------------------------

model = RandomForestRegressor(
    n_estimators=30,
    max_depth=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=2
)

model.fit(X_train_imp, y_train)

# -----------------------------
# Predict
# -----------------------------

predictions = model.predict(X_test_imp)

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))

print("\nMODEL PERFORMANCE")
print("=================")
print("MAE:", round(mae, 4))
print("RMSE:", round(rmse, 4))

Training matrix: (200000, 14)
Testing matrix: (100000, 14)

MODEL PERFORMANCE
MAE: 0.0285
RMSE: 0.2601


In [9]:
print("model_page exists:", "model_page" in globals())
print("rf_model exists:", "rf_model" in globals())

model_page exists: False
rf_model exists: False


In [10]:
print("model exists:", "model" in globals())
print("predictions exists:", "predictions" in globals())
print("test_model exists:", "test_model" in globals())
print("features exists:", "features" in globals())

model exists: True
predictions exists: True
test_model exists: True
features exists: True


In [11]:
rf_model = model

print("Random Forest model saved as rf_model:", "rf_model" in globals())

Random Forest model saved as rf_model: True


In [12]:
# Create page-level model predictions

test_results = test_model[
    ["content_hash_id", "sessions_organic"]
].copy()

test_results["predicted_organic_sessions"] = predictions

model_page = (
    test_results
    .groupby("content_hash_id", as_index=False)
    .agg(
        actual_organic_sessions=("sessions_organic", "sum"),
        predicted_organic_sessions=("predicted_organic_sessions", "sum")
    )
)

model_page["ml_rank"] = (
    model_page["predicted_organic_sessions"]
    .rank(method="min", ascending=False)
)

print("Pages in model ranking:", len(model_page))

display(
    model_page
    .sort_values(
        "predicted_organic_sessions",
        ascending=False
    )
    .head(10)
)

Pages in model ranking: 87711


,content_hash_id,actual_organic_sessions,predicted_organic_sessions,ml_rank
60153,content_afcca85076bb17d3,37.0,36.945842,1.0
10200,content_1dbd310eb57d2182,34.0,33.723524,2.0
29984,content_57c3b90b328b406e,43.0,33.242233,3.0
10896,content_1fb7c75c360b4a7f,32.0,31.817365,4.0
79601,content_e8a52cf3d5988c07,70.0,31.275361,5.0
8048,content_17494d099b0a537e,29.0,31.143729,6.0
29875,content_57768353f230d65d,27.0,30.481188,7.0
40750,content_76f9b5358889447c,22.0,30.195011,8.0
6890,content_13bbfc7e0f401f59,30.0,29.638118,9.0
8837,content_19b310673809a9ee,39.0,29.204289,10.0


In [5]:
# Recreate the Week-4 baseline from the same March data

baseline_page = (
    df.groupby("content_hash_id", as_index=False)
      .agg(
          organic_sessions=("sessions_organic", "sum"),
          gsc_impressions=("gsc_impressions", "sum"),
          gsc_clicks=("gsc_clicks", "sum")
      )
)

baseline_page["ctr"] = np.where(
    baseline_page["gsc_impressions"] > 0,
    baseline_page["gsc_clicks"] /
    baseline_page["gsc_impressions"],
    np.nan
)

traffic_threshold = baseline_page["organic_sessions"].quantile(0.25)

ctr_threshold = baseline_page.loc[
    baseline_page["gsc_impressions"] >= 100,
    "ctr"
].quantile(0.25)

baseline_page["low_organic_traffic"] = (
    baseline_page["organic_sessions"] <= traffic_threshold
)

baseline_page["low_ctr"] = (
    (baseline_page["gsc_impressions"] >= 100) &
    (baseline_page["ctr"] <= ctr_threshold)
)

baseline_page["baseline_score"] = (
    baseline_page["low_organic_traffic"].astype(int) * 2
    +
    baseline_page["low_ctr"].astype(int) * 2
)

baseline_page["baseline_rank"] = (
    baseline_page["baseline_score"]
    .rank(method="min", ascending=False)
)

print("Baseline pages:", len(baseline_page))

Baseline pages: 331437


In [13]:
from scipy.stats import spearmanr

comparison = model_page.merge(
    baseline_page[
        ["content_hash_id", "baseline_score", "baseline_rank"]
    ],
    on="content_hash_id",
    how="inner"
)

spearman_corr, p_value = spearmanr(
    comparison["ml_rank"],
    comparison["baseline_rank"]
)

print("MODEL VS BASELINE")
print("=================")
print("Pages compared:", len(comparison))
print("Spearman rank correlation:", round(spearman_corr, 4))
print("P-value:", round(p_value, 4))

print("\nTop 10 ML-ranked pages:")
display(
    comparison
    .sort_values("predicted_organic_sessions", ascending=False)
    .head(10)
)

MODEL VS BASELINE
Pages compared: 87711
Spearman rank correlation: -0.3513
P-value: 0.0

Top 10 ML-ranked pages:


,content_hash_id,actual_organic_sessions,predicted_organic_sessions,ml_rank,baseline_score,baseline_rank
60153,content_afcca85076bb17d3,37.0,36.945842,1.0,0,289315.0
10200,content_1dbd310eb57d2182,34.0,33.723524,2.0,0,289315.0
29984,content_57c3b90b328b406e,43.0,33.242233,3.0,0,289315.0
10896,content_1fb7c75c360b4a7f,32.0,31.817365,4.0,0,289315.0
79601,content_e8a52cf3d5988c07,70.0,31.275361,5.0,0,289315.0
8048,content_17494d099b0a537e,29.0,31.143729,6.0,0,289315.0
29875,content_57768353f230d65d,27.0,30.481188,7.0,0,289315.0
40750,content_76f9b5358889447c,22.0,30.195011,8.0,0,289315.0
6890,content_13bbfc7e0f401f59,30.0,29.638118,9.0,0,289315.0
8837,content_19b310673809a9ee,39.0,29.204289,10.0,0,289315.0


In [14]:
print("df exists:", "df" in globals())
print("baseline_page exists:", "baseline_page" in globals())
print("model_page exists:", "model_page" in globals())
print("rf_model exists:", "rf_model" in globals())

df exists: True
baseline_page exists: True
model_page exists: True
rf_model exists: True


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## 4. Errors and interpretation

The error analysis focuses on where predicted organic sessions differ most from observed organic sessions in the held-out period.

Large errors do not automatically indicate a bad page or a bad model. They can occur because traffic is highly variable, because some pages have sparse observations, or because important factors are not present in this warehouse.

I will also inspect Random Forest feature importance to understand which observed signals the model relied on most.

Feature importance shows association used by the model; it does not establish causation.

The final interpretation will therefore remain directional and decision-support oriented.

In [16]:
# Generate predictions for the held-out test set

y_pred = model.predict(X_test)

print("Predictions created:", len(y_pred))
print("Actual test values:", len(y_test))

Predictions created: 100000
Actual test values: 100000


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(


In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-08 Section 4
# Error analysis

import pandas as pd
import numpy as np

error_analysis = pd.DataFrame({
    "observed": np.asarray(y_test),
    "predicted": np.asarray(y_pred)
})

error_analysis["error"] = (
    error_analysis["predicted"] -
    error_analysis["observed"]
)

error_analysis["absolute_error"] = (
    error_analysis["error"].abs()
)

print("ERROR ANALYSIS")
print("==============")
print("Test observations:", len(error_analysis))
print(
    "Mean absolute error:",
    round(error_analysis["absolute_error"].mean(), 4)
)

print("\nLargest prediction errors:")

display(
    error_analysis
    .sort_values("absolute_error", ascending=False)
    .head(10)
)

overpredict = (error_analysis["error"] > 0).sum()
underpredict = (error_analysis["error"] < 0).sum()
exact = (error_analysis["error"] == 0).sum()

print("\nPREDICTION DIRECTION")
print("====================")
print("Overpredictions:", overpredict)
print("Underpredictions:", underpredict)
print("Exact predictions:", exact)

ERROR ANALYSIS
Test observations: 100000
Mean absolute error: 0.0285

Largest prediction errors:


,observed,predicted,error,absolute_error
89680,70.0,31.275361,-38.724639,38.724639
27103,23.0,5.321032,-17.678968,17.678968
49650,31.0,14.485493,-16.514507,16.514507
59160,39.0,29.204289,-9.795711,9.795711
99782,43.0,33.242233,-9.757767,9.757767
17787,16.0,6.579240,-9.420760,9.420760
59150,12.0,3.131474,-8.868526,8.868526
25902,24.0,15.448750,-8.551250,8.551250
43630,22.0,30.195011,8.195011,8.195011
78315,27.0,19.404453,-7.595547,7.595547



PREDICTION DIRECTION
Overpredictions: 2048
Underpredictions: 2364
Exact predictions: 95588


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.